[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/en/lab11/lab11.ipynb)

# Lab 11: Probabilistic graphical models

In this lab we will develop Python code to represent **Bayesian networks**. We will be able to store models, learn them by *MLE (maximum likelihood estimation)* and use them to make inferences.

## Representation of variables

The models will work exclusively with categorical variables. To represent these variables we will create a class that stores the name of a variable and its possible values.

We will also include a method that allows us to encode each possible value with an integer.

In [ ]:
class Variable:
  def __init__(self, name, values):
    self.name = name
    self.values = values

  def encode(self, valor):
    # TODO: Complete the method
    return ...

  def __str__(self):
    return f'Variable: {self.name} ({"|".join(self.values)})'

# Checks
# TODO: Create a height variable with possible values: baja, media, alta
height_var = ...
assert(height_var.encode('medium')==1)

## The graph

Bayesian networks are **directed acyclic graphs** whose nodes represent the variables covered by the model.

### Nodes

We will represent these nodes with the Node class, which will record the variable they represent and will have two lists, one of child nodes and another of parent nodes.

In addition, the nodes will have a method to indicate whether the node is a root, that is, whether it has no parents.

In [ ]:
class Node:
  def __init__(self, variable):
    self.variable = variable
    self.parents = []
    self.children = []

  def add_parent(self, p):
    # TODO: Complete the method
    ...

  def add_child(self, p):
    # TODO: Complete the method
    ...

  def is_root(self):
    # TODO: Complete the method
    ...

  def __str__(self):
    r = f'Node: {self.variable.name}'
    r += ' P:['+' & '.join(map(lambda x: x.variable.name,self.parents))+']'
    r += ' H:['+' & '.join(map(lambda x: x.variable.name,self.children))+']'
    return r

# Checks
height_node = Node(height_var)
print(height_node)
assert(height_node.is_root())

### Representing the directed graph

To represent the graph we will have the Graph class, which will simply be a list of nodes (properly linked to each other). A graph will be created from a list of variables and a list of edges (tuples (source_variable, target_variable)).

Complete the code to represent the following graph.

<img src="./img/grafo.png" alt="Graph to represent" width="700"/>

In [ ]:
class Graph:
  def __init__(self, variables, edges):
    self.nodes = [Node(var) for var in variables]

    # Auxiliary dict so we can refer to nodes by variable name
    # without searching the list
    nodes_dict = {node.variable.name:node for node in self.nodes}
    for inicio,fin in edges:
      # TODO: Represent the edge by adding parents/children to the nodes
      ...

  def print_graph(self):
    for node in self.nodes:
      print(node)


diet_var = Variable('diet',['bad','good'])
weight_var = Variable('weight',['low','high'])
bmi_var = Variable('BMI',['low','high'])
# TODO: Declare the requested graph
height_graph = ...

# Checks - verify the graph is represented correctly
height_graph.print_graph()

## (Conditional) probability distributions

For the graph to constitute a Bayesian network, each node must be accompanied by a **probability distribution** of the variable $X$ it represents (conditioned on the variables of its parent nodes, if any):

$$P(X_i | parents(X_i))$$

These probability distributions will contain parameters that tell us the probability that the variable $X$ takes a specific value given the values of each of its parents. We will need to represent these probability distributions, so for this we will declare the CPD class. This class will store:
1. The variable it represents
1. The list of conditioning variables
1. The parameters of the CPD. We will store them in a table that will have a row per value of $X$ and a row per combination of values of the conditioning variables (as in the following example).

<img src="./img/cpd.png" alt="CPD example" width="400"/>

There will be two ways to initialize the parameters: either by receiving a NumPy array of the appropriate dimensions, or by initializing them all to 0 (this will be the default behavior).

Finally, we will include a method that, given a row with data containing values for the variables involved in this CPD, returns the corresponding probability value.

In [ ]:
import numpy as np
import pandas as pd

class CPD:
  def __init__(self, var, vars_condicion, values = None):
    self.variable = var
    self.conditioning_variables = vars_condicion

    # TODO: Compute the parameter-matrix shape
    dimensiones = ...
    if values is not None and values.shape==dimensiones:
      self.cpd = values
    else:
      # TODO: Initialize the parameter matrix
      self.cpd = ...
  
  def get_prob(self, data):
    # TODO: Return the correct value (use the two helper methods below)
    return ...

  def get_column_index(self, data):
    '''
    Given an item with values for the involved variables,
    return the corresponding column index in the parameter matrix
    '''
    return self.variable.encode(data[self.variable.name])

  def get_row_index(self, data):
    '''
    Given an item with values for the involved variables,
    return the corresponding column index in the parameter matrix
    '''
    parents_index = 0
    block_size = 1
    for v in self.conditioning_variables:
      parents_index += block_size*v.encode(data[v.name])
      block_size *= len(v.values)
    return parents_index

  def __str__(self):
    '''
    Helper to print the formatted CPD
    '''
    df = pd.DataFrame(self.cpd, columns=self.variable.values)
    cond = ''
    if len(self.conditioning_variables)>0:
      cond = '|' + ','.join(map(lambda x:x.name, self.conditioning_variables))
    df.insert(loc=0,column=f'P({self.variable.name}{cond})', value=self.name_rows())
    return df.to_string(index=False)

  def name_rows(self):
    '''
    Helper to name the rows of the parameter matrix
    '''
    nombres = ['']
    for v in self.conditioning_variables:
      nombres_original = nombres
      nombres = []
      for val in v.values:
        nombres += [val+','+n for n in nombres_original]
    return nombres

# Checks
weight_cpd = CPD(weight_var, [height_var, diet_var])
assert(weight_cpd.get_prob({'height':'low', 'diet':'good', 'weight':'low'})==0)
weight_cpd = CPD(weight_var, [height_var, diet_var], np.array([[0.1,0.9],[0.2,0.8],[0.3,0.7],[0.4,0.6],[0.45,0.55],[0.47,0.53]]))
assert(weight_cpd.get_prob({'height':'low', 'diet':'good', 'weight':'low'})==0.4)

# Bayesian network

Once we have representations for graphs and CPDs, we can move on to representing our Bayesian network model. We will define the BayesianNetwork class that will store the graph and the CPDs (the latter in a dictionary indexed by the name of the variables, to facilitate access).

The network will be created from a graph and will initially create CPDs with parameters set to 0 for each variable.

In [ ]:
class BayesianNetwork:
  def __init__(self, graph):
    self.graph = graph
    self.cpds = {}
    for node in graph.nodes:
      # TODO: Create a CPD with the correct variables (parameters at 0) for node.variable
      ...

est_network = BayesianNetwork(height_graph)

## Training by Maximum Likelihood

One of the ways to learn the parameters of the CPDs associated with a Bayesian network is to use **maximum likelihood estimation**. It consists of finding the parameters of the CPDs that maximize the probability assigned to all the examples contained in the training dataset:

$$P(X|θ) = \prod_i P(x^{(i)}|\theta)$$

The chain rule and the mathematical development of the formula tell us that each parameter can be obtained by a mere count.

We must adapt our CPD class so that it allows accumulating the counts during the training phase and, once training is finished, normalizes the values of the tables so that they represent probability distributions.

We can take advantage of the dynamic nature of Python to add these new functionalities without having to re-run the previous cells.

In [ ]:
def count(self, data):
  '''
  CPD method
  Increment by 1 the parameter matching the values in data
  '''
  # TODO: Complete the method (remember get_row_index and get_column_index)
  ...

def normalize(self):
  '''
  CPD method
  Normalize the parameter matrix (which stores counts during training)
  so that it represents probability distributions
  '''
  # TODO: Normalize the CPD matrix. Each row must sum to 1.
  self.cpd /= ...

# Bind the methods to the class to add the new behaviour
CPD.count = count
CPD.normalize = normalize

# Checks
example_data = [{'weight':'low', 'diet':'good'},\
                 {'weight':'low', 'diet':'bad'},\
                 {'weight':'high', 'diet':'good'},\
                 {'weight':'high', 'diet':'bad'},\
                 {'weight':'high', 'diet':'bad'},\
                 {'weight':'high', 'diet':'bad'}]

weight_cpd = CPD(weight_var, [diet_var])
for d in example_data:
  weight_cpd.count(d)
assert(weight_cpd.get_prob({'weight':'high', 'diet':'good'})==1)
assert(weight_cpd.get_prob({'weight':'high', 'diet':'bad'})==3)
weight_cpd.normalize()
print(weight_cpd)
assert(weight_cpd.get_prob({'weight':'high', 'diet':'good'})==0.5)
assert(weight_cpd.get_prob({'weight':'low', 'diet':'bad'})==0.25)

With this adaptation, we can now write the method to perform maximum likelihood training. The method will receive a Pandas DataFrame and will iterate once over the rows doing the counts.

In [ ]:
def fit(self, data):
  '''
  BayesianNetwork method
  Run MLE training from the data received as argument
  
  Params:
    data: DataFrame of examples described by the variables covered by the network
  '''
  for _, row in data.iterrows():
    for n in self.graph.nodes:
      # TODO: Update the corresponding CPD count
      ...

  # TODO: Normalize all CPDs
  ...

BayesianNetwork.fit=fit

# Checks
test_network = BayesianNetwork(Graph([diet_var, weight_var],[(diet_var,weight_var)]))
test_network.fit(pd.DataFrame(example_data))
assert(test_network.cpds['weight'].get_prob({'weight':'low', 'diet':'bad'})==0.25)

## MAP prediction (Maximum a posteriori)

Once we have a trained network, we can estimate the probability of a given example in which all its variables have an assigned value ($P(x^{(k)}_0,x^{(k)}_1,...,x^{(k)}_d)$) using the chain rule. The probability will be the product of the probabilities assigned in the different CPDs of the network.

$$P(\mathbf{x}^{(k)}) = \prod_i P(x^{(k)}_i | parents(x^{(k)}_i))$$

We will also add the functionality of making predictions of a variable $Y$ from the observation of the others ($P(Y | (X-Y))$). To do this we will estimate the probability of the example with all the possible values of $Y$ and return the value with the highest probability.

$$MAP(Y | (X-Y)=(\mathbf{x}-\mathbf{y})^{(k)})=argmax_{y}(P(Y=y | (X-Y)=(\mathbf{x}-\mathbf{y})^{(k)}))$$

In [ ]:
def get_prob(self, row):
  '''
  BayesianNetwork method
  Given a data row, return its probability.
  The row must include the parents of every variable it contains.
  '''
  # TODO: Compute the probability using the chain rule
  p = 1
  ...
  return p

def predict_probs(self, row, target_variable):
  '''
  BayesianNetwork method
  Given a data row, return a list with the probability of each possible
  value of the target variable

  Returns:
    List of (probability, value) tuples
  '''
  probs = []
  # Copy so we can change the value without affecting the original data
  values = row.copy()

  # TODO: Append the required tuples to probs
  ...
  return probs

def predict(self, row, target_variable):
  '''
  BayesianNetwork method
  Given a data row, return the most probable value of the target variable
  '''
  probs = self.predict_probs(row, target_variable)
  # TODO: Return the highest-probability value
  ...


BayesianNetwork.get_prob=get_prob
BayesianNetwork.predict_probs=predict_probs
BayesianNetwork.predict=predict

# Checks
print(test_network.predict_probs({'weight':'high'},diet_var))
assert(test_network.predict({'weight':'high'},diet_var)=='bad')

# Experiments

## Dataset

To test our model we will download the *Nursery* dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Nursery). This dataset represents applications to a Slovenian nursing school. Each application presents a series of (categorical) sociodemographic characteristics and is assessed with a decision that indicates its priority for admission.

In [ ]:
from os import listdir
import os.path

PATH = 'lab11/'

if not os.path.exists(PATH):
    os.mkdir(PATH)
    !wget https://archive.ics.uci.edu/ml/machine-learning-databases/nursery/nursery.data -O lab11/nursery.data

variable_names = ['parents','has_nurs','form','children','housing','finance','social','health','decision']
LABEL_NAME = 'decision'

data = pd.read_csv(PATH + 'nursery.data', ',', names=variable_names)

# We create a Variable object for each variable where we store all its
# posibles values
variables = []
label = None
for v in variable_names:
  variables.append(Variable(v,list(data[v].unique())))
  if v==LABEL_NAME:
    label = Variable(v,list(data[v].unique()))

# We display the data for a first superficial look
print(data)

We make a train-test split using the *sklearn* library.

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(data, test_size=0.3, random_state=1235)

# Naïve-Bayes model

The Bayesian network we are going to try will be a **Naïve Bayes** model. In this model, the independence of all input variables is assumed, while it is assumed that the probability of each variable depends on the class.

<img src="./img/naive-bayes.png" alt="Naive Bayes model architecture" width="700"/>

Create the corresponding Bayesian network and train it with *train_data*

In [ ]:
# TODO: Crea el Graph y la BayesianNetwork
naive_network = ...

# TODO: Train the network
...

### Model verification

To check the effectiveness of our model we are going to measure the classification accuracy and the F1-score of the predictions on the test set. To do this we will use the sklearn library.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# TODO: Get predictions on the test set
predictions = ...

print('Accuracy:',accuracy_score(test_data[label.name], predictions))
print('F1-score:',f1_score(test_data[label.name], predictions, average='macro'))

## Other uses for the model

An advantage of Bayesian networks is that they can be used to generate data that follow the distribution they represent. Let's include this functionality in our BayesianNetwork. Generating a new example will consist of assigning a random value (respecting the probabilities indicated by the CPD of that variable) to each variable. We should start with the variables that are not conditioned and use the assigned values to generate the successive values.

In [ ]:
def sample(self, values):
  '''
  CPD method
  Assign a random value to the variable according to the CPD parameters and
  the values received as argument
  '''
  row_index = self.get_row_index(values)
  r = np.random.random()
  for i,p in enumerate(self.cpd[row_index]):
    if r < p:
      return self.variable.values[i]
    r -= p
  return self.variable.values[-1]

CPD.sample=sample

def sample(self):
  '''
  BayesianNetwork method
  Create a new dictionary with a value assigned to each variable covered
  by the network
  '''
  sampled_values = {} # Generated sample

  # Pick the next unsampled CPD for which we have all conditioning-variable values
  while len(sampled_values)<len(self.cpds):
    next_cpd = None
    for var_name,cpd in self.cpds.items():
      if var_name not in sampled_values:
        cpd_valid = True
        for condition in cpd.conditioning_variables:
          if condition.name not in sampled_values:
            cpd_valid = False
            continue
        if cpd_valid:
            next_cpd = cpd
    
    # Sample the CPD
    sampled_values[next_cpd.variable.name] = next_cpd.sample(sampled_values)
  return sampled_values

def sample_dataframe(self, num_samples):
  '''
  BayesianNetwork method
  Create a new DataFrame of num_samples rows following the distribution
  represented by the network
  '''
  elems = []
  for i in range(num_samples):
    elems.append(self.sample())
  return pd.DataFrame(elems)

BayesianNetwork.sample=sample
BayesianNetwork.sample_dataframe=sample_dataframe

# Checks
print(naive_network.sample_dataframe(10))

# Exercises
1. Create the network of the image (use the supplied values) and generate 10000 examples in a DataFrame. Export it to CSV and repeat the experiment to check how well the Naïve Bayes model works in that problem.
1. Create again the same network, but with the parameters at 0. Train it with the generated CSV. Do you expect the Accuracy and F1-Score results? Are they as good as you expected?

<img src="./img/red-ejemplo.png" alt="Naive Bayes model architecture" width="700"/>

In [ ]:
LABEL_NAME = 'letter'

variables_v = {'difficulty': ['easy', 'hard'],'intelligence': ['low','high'],'grade':['fail','pass','excellent'],'ebau':['fail','pass'],'letter':['no','yes']}

cpd_difficulty = np.array([[0.6, 0.4]])
cpd_intelligence = np.array([[0.7, 0.3]])
cpd_grade = np.array([[0.3, 0.4, 0.3],[0.7,0.25,0.05],[0.02,0.08,0.9],[0.2,0.3,0.5]])
ebau_cpd = np.array([[0.95, 0.05],[0.2,0.8]])
letter_cpd = np.array([[0.99, 0.01],[0.4,0.6],[0.1,0.9]])